# 04 - ML Forecast Features (Gold)

One row per province, crop, year, and quarter.
`production_next_quarter` is the label only. It is not a model feature.
Same-quarter `growth_rate_yoy` and `risk_label` are excluded (target leakage).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from configs.project_config import SILVER_TABLES, GOLD_TABLES

source = SILVER_TABLES['crop_production_enriched']
target = GOLD_TABLES['ml_crop_forecast_features']

df = spark.table(source)

w = Window.partitionBy('province', 'crop').orderBy('time_index')

features = (
    df.withColumn('production_next_quarter', F.lead('production', 1).over(w))
    .select(
        F.col('canonical_province').alias('province'),
        'crop',
        'crop_group',
        'year',
        'quarter',
        'quarter_num',
        'time_index',
        'production',
        'production_next_quarter',
        'production_lag_1',
        'production_lag_2',
        'production_lag_4',
        'temperature_2m_mean',
        'temperature_2m_max',
        'temperature_2m_min',
        'precipitation_sum',
        'rain_sum',
        'precipitation_hours',
        'sunshine_duration',
        'reference_evapotranspiration_mm',
        'wind_speed_10m_max',
        'wind_gusts_10m_max',
        'rainfall_deviation_pct',
        'temperature_anomaly_c',
        'temp_anomaly_rolling',
        'rain_anomaly_rolling',
        'oni_index',
        'enso_phase_la_nina',
        'enso_phase_neutral'
    )
)

(
    features.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(target)
)

out = spark.table(target)
labeled = out.filter(F.col('production_next_quarter').isNotNull()).count()
print(f'Wrote: {target}')
print(f'Rows : {out.count()}')
print(f'Rows with a next-quarter label: {labeled}')
print(f'Cols : {out.columns}')
